# First look: ข้อมูล Air4Thai

**จุดประสงค์ของ notebook นี้:** ดูโครงสร้างข้อมูล ช่วงเวลา หน่วย และค่าที่หายไป **ยังไม่วิเคราะห์หรือสรุปผล**

1. ข้อมูลปัจจุบันจาก Air4Thai JSON endpoint
2. ข้อมูลตัวอย่างรายชั่วโมงของรายวิชา (`data/sample/`)

## 1. ข้อมูลปัจจุบันจาก Air4Thai

In [1]:
import json
import os

import pandas as pd
import requests
import truststore

truststore.inject_into_ssl()

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
try:
    r = requests.get(URL, timeout=30)
    print("status:", r.status_code, "| content-type:", r.headers.get("Content-Type"))
    data = r.json()
    source = "live API"
except Exception as e:
    # ถ้าดึงสดไม่ได้ ใช้ไฟล์ตัวอย่างของรายวิชาที่มีรูปแบบ response เดียวกัน
    print("ดึงข้อมูลสดไม่สำเร็จ:", type(e).__name__, "→ ใช้ ../data/sample/air4thai_sample.json แทน")
    with open("../data/sample/air4thai_sample.json", encoding="utf-8") as f:
        data = json.load(f)
    source = "sample file"
print("แหล่งข้อมูลที่ใช้:", source)

status: 200 | content-type: application/json; charset=UTF-8
แหล่งข้อมูลที่ใช้: live API


In [2]:
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]          # ถ้า key ไม่ชื่อนี้ ให้แก้ตามที่เห็นจาก cell ก่อนหน้า
print(len(stations), "สถานี")
stations[0]                          # ดูโครงสร้างของสถานีแรกทั้งก้อน

<class 'dict'> ['stations']
180 สถานี


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-24',
  'time': '09:00',
  'PM25': {'color_id': '1', 'aqi': '11', 'value': '6.8'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '8', 'value': '11'},
  'CO': {'color_id': '1', 'aqi': '4', 'value': '0.70'},
  'NO2': {'color_id': '1', 'aqi': '0', 'value': '1'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '1', 'aqi': '11', 'param': 'PM25'}}}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()              # ชื่อคอลัมน์ที่ซ้อนกันจะต่อด้วยจุด

(180, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

### คำตอบจากการดูโครงสร้าง

- **หนึ่งแถวคืออะไร:** หนึ่งสถานีตรวจวัด กับค่าล่าสุดของสถานีนั้น (หนึ่งเวลา)
- **คอลัมน์สำคัญ:** รหัสสถานี `stationID`, ชื่อ `nameTH` / `nameEN`, พิกัด `lat` / `long`, วันที่ `AQILast.date`, เวลา `AQILast.time`
- **PM2.5 อยู่ที่ไหน:** `AQILast.PM25.value` คือความเข้มข้น (µg/m³) และ `AQILast.PM25.aqi` คือค่าดัชนี AQI มีทั้งสองแบบ
- **ชนิดข้อมูล:** ค่าตัวเลขเก็บเป็นข้อความ (string) ทั้งหมด รวมถึง `lat` / `long` ต้องแปลงก่อนใช้
- **ค่าแปลก:** มีค่า `-1` ซึ่งไม่ใช่ความเข้มข้นจริง น่าจะแทนข้อมูลหาย และบางสถานีมีวันที่ล่าสุดเก่ากว่าสถานีอื่น ควรตรวจสอบต่อ

In [4]:
PM25_COL = "AQILast.PM25.value"
pm = pd.to_numeric(df_now[PM25_COL], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
print("วันที่ล่าสุดของแต่ละสถานี:")
print(df_now["AQILast.date"].value_counts().head())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 2
วันที่ล่าสุดของแต่ละสถานี:
AQILast.date
2026-11-24    175
2026-10-21      1
2026-10-09      1
2026-11-11      1
2026-11-18      1
Name: count, dtype: int64


count    180.000000
mean      22.259444
std        8.633834
min       -1.000000
25%       16.425000
50%       22.400000
75%       28.925000
max       47.200000
Name: AQILast.PM25.value, dtype: float64

ข้อมูลปัจจุบันมีเพียงหนึ่งเวลาต่อสถานี จึงยังตรวจความถี่และช่วงเวลาไม่ได้ ดูในส่วนที่ 2

## 2. ข้อมูลตัวอย่างรายชั่วโมง (ช่วงสำรวจเท่านั้น)

In [5]:
print("working directory:", os.getcwd())
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ใช้เฉพาะช่วงสำรวจ 2563–2566 (ค.ศ. 2020–2023) ห้ามดูช่วงทดสอบ 2567–2568
explore = hourly[hourly["ts"] < "2024-01-01"]
print("แถวในช่วงสำรวจ:", explore.shape)
print("ช่วงเวลาของไฟล์ตัวอย่างทั้งหมด:", hourly["ts"].min(), "→", hourly["ts"].max())

working directory: D:\pm25-explorer-0277\pm25-explorer-0277\notebooks


แถวในช่วงสำรวจ: (0, 8)
ช่วงเวลาของไฟล์ตัวอย่างทั้งหมด: 2026-09-01 00:00:00 → 2026-11-17 23:00:00


**ข้อสังเกต:** ไฟล์ตัวอย่างไม่มีข้อมูลช่วง 2563–2566 เลย (0 แถว) เพราะมีเฉพาะ ก.ย.–พ.ย. 2569
ตาม README ของ `sample-data/` ระบุว่า API ย้อนหลังของ Air4Thai เก็บข้อมูลได้ประมาณ 3 เดือน

ปี 2569 ไม่ได้อยู่ในช่วงทดสอบ 2567–2568 จึงใช้ข้อมูลส่วนนี้ดู **โครงสร้าง** แทน โดยยังไม่ได้ดูช่วงทดสอบ

In [6]:
sample = hourly  # ทั้งไฟล์คือปี 2569 ไม่ทับช่วงทดสอบ
print(sample.shape, sample["station_id"].nunique(), "สถานี")
print(sample["ts"].min(), "→", sample["ts"].max())
# ความถี่ของข้อมูล คิดภายในแต่ละสถานี (ควรเป็น 1 ชั่วโมง)
print(sample.sort_values(["station_id", "ts"]).groupby("station_id")["ts"].diff().value_counts().head(3))
sample.isna().mean().round(3)

(333216, 8) 178 สถานี
2026-09-01 00:00:00 → 2026-11-17 23:00:00
ts
0 days 01:00:00    333038
Name: count, dtype: int64


station_id    0.000
ts            0.000
pm25          0.071
pm10          0.646
o3            0.849
co            0.811
no2           0.794
so2           0.828
dtype: float64

In [7]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
missing_by_station = (
    sample.groupby("station_id")["pm25"]
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
)
(missing_by_station.head(10).rename("pm25_missing").reset_index()
    .merge(st[["station_id", "name_th", "province", "region"]], on="station_id"))

,station_id,pm25_missing,name_th,province,region
0,o67,0.986645,วัดพระพุทธบาท ราชวรมหาวิหาร,สระบุรี,กลาง
1,o29,0.942842,ที่ว่าการอำเภอแม่สะเรียง,แม่ฮ่องสอน,เหนือ
2,o59,0.933226,การไฟฟ้าส่วนภูมิภาค สาขา อำเภอฮอด,เชียงใหม่,เหนือ
3,o64,0.923077,ท่าอากาศยานปาย,แม่ฮ่องสอน,เหนือ
4,o68,0.905449,ศูนย์ป้องกันและบรรเทาสาธารณภัยเทศบาลตำบลเวียง,เชียงราย,เหนือ
5,76t,0.803419,ศูนย์การศึกษานอกโรงเรียน,ตาก,ตะวันตก
6,74t,0.748932,ศูนย์ราชการจังหวัดระยอง,ระยอง,ตะวันออก
7,18t,0.705128,ศาลากลางจังหวัดสมุทรปราการ,สมุทรปราการ,กลาง
8,75t,0.704060,โรงพยาบาลเฉลิมพระเกียรติ,น่าน,เหนือ
9,26t,0.618056,สำนักงานสิ่งแวดล้อมภาคที่8,ราชบุรี,ตะวันตก


In [8]:
# ค่าหายและค่ากลางของ PM2.5 แยกตามภาค (ดูเท่านั้น)
by_region = sample.merge(st[["station_id", "region"]], on="station_id")
by_region.groupby("region")["pm25"].agg(
    mean="mean", median="median", missing=lambda s: s.isna().mean()
).round(3)

,mean,median,missing
region,,,
กลาง,16.594,15.1,0.035
ตะวันตก,15.590,13.1,0.450
ตะวันออก,15.020,12.5,0.128
ตะวันออกเฉียงเหนือ,12.737,11.2,0.031
เหนือ,12.751,11.9,0.245
ใต้,11.017,8.6,0.003


### สรุปสิ่งที่เห็น (ยังไม่ใช่ข้อสรุป)

1. พบว่าไฟล์ตัวอย่างมีเฉพาะ ก.ย.–พ.ย. 2569 ไม่มีช่วงสำรวจ 2563–2566 ควรตรวจสอบต่อว่าจะหาข้อมูลย้อนหลังจากแหล่งใด
2. พบว่าข้อมูลรายชั่วโมงห่างกัน 1 ชั่วโมงสม่ำเสมอ และทุกสถานีมีจำนวนแถวเท่ากัน แต่ค่า PM2.5 หายราว 7% ควรตรวจสอบต่อว่าหายเป็นช่วงยาวหรือกระจาย
3. พบว่าสถานีที่ค่า PM2.5 หายมากที่สุดหลายแห่งอยู่ในภาคเหนือ บางสถานีหายเกิน 90% ควรตรวจสอบต่อก่อนใช้สถานีเหล่านั้น
4. พบว่า API ค่าปัจจุบันใช้ `-1` แทนค่าหาย และเก็บตัวเลขเป็นข้อความ ควรแปลงและกรองก่อนคำนวณ
5. พบว่าค่ากลาง PM2.5 ภาคกลางสูงกว่าภาคใต้ในช่วงนี้ แต่เป็นข้อมูลแค่ 3 เดือนนอกฤดูฝุ่น ควรตรวจสอบต่อในช่วงทดสอบ